# Memory Store
> In-memory session store

In [ ]:
#| default_exp store.memory

In [ ]:
#| export
from typing import Optional, Dict, List
from memexcode.core import gen_id, now_iso, MessageEntry, CompactionEntry
from memexcode.session import SessionEntry
from memexcode.pluginspec import SessionABC, SessionStore, SessionInfo, TreeNode, hookimpl

In [ ]:
#| export
class MemorySession(SessionABC):
    def __init__(self):
        self._entries: List[SessionEntry] = []
        self._by_id: Dict[str, SessionEntry] = {}

    async def append(self, entry: SessionEntry) -> None:
        if not entry.id:
            entry._entry.id = gen_id()
        if self._entries:
            entry._entry.parentId = self._entries[-1].id
        self._entries.append(entry)
        self._by_id[entry.id] = entry

    async def get_by_id(self, entry_id: str) -> Optional[SessionEntry]:
        return self._by_id.get(entry_id)

    async def get_path_to_root(self, leaf_id: str) -> List[SessionEntry]:
        path = []
        current = self._by_id.get(leaf_id)
        while current:
            path.insert(0, current)
            current = self._by_id.get(current.parentId) if current.parentId else None
        return path

    async def load_all(self) -> List[SessionEntry]:
        return list(self._entries)

    async def compact(self, first_kept_id: str, summary: str) -> None:
        idx = next((i for i, e in enumerate(self._entries) if e.id == first_kept_id), None)
        if idx is None:
            raise ValueError(f'first_kept_id {first_kept_id} not found')
        parent = self._entries[idx].parentId if idx > 0 else None
        compact = SessionEntry(CompactionEntry(
            parentId=parent, summary=summary, firstKeptEntryId=first_kept_id
        ))
        self._entries = [compact] + self._entries[idx:]
        self._by_id = {e.id: e for e in self._entries}

    async def get_tree(self, root_id: Optional[str] = None) -> TreeNode:
        root = self._entries[0] if not root_id else self._by_id.get(root_id)
        if not root:
            return TreeNode(SessionEntry(MessageEntry()), [])
        children = {e.id: [] for e in self._entries}
        for e in self._entries:
            if e.parentId and e.parentId in children:
                children[e.parentId].append(e)
        def build(entry):
            return TreeNode(entry, [build(c) for c in children.get(entry.id, [])])
        return build(root)

class MemoryStore(SessionStore):
    def __init__(self):
        self._sessions: Dict[str, MemorySession] = {}
        self._infos: Dict[str, SessionInfo] = {}

    def get_session(self, session_id: str) -> SessionABC:
        return self._sessions[session_id]

    async def create(self, cwd: str, name: Optional[str] = None, parent_session: Optional[str] = None) -> SessionInfo:
        info = SessionInfo(id=gen_id(), cwd=cwd, created_at=now_iso(), updated_at=now_iso(),
                          name=name, parent_session=parent_session)
        self._sessions[info.id] = MemorySession()
        self._infos[info.id] = info
        return info

    async def list(self) -> List[SessionInfo]:
        return list(self._infos.values())

    async def rename(self, session_id: str, name: str) -> None:
        if session_id in self._infos:
            self._infos[session_id].name = name
            self._infos[session_id].updated_at = now_iso()

    async def delete(self, session_id: str) -> None:
        self._sessions.pop(session_id, None)
        self._infos.pop(session_id, None)

    async def fork(self, session_id: str, leaf_id: Optional[str] = None, name: Optional[str] = None) -> SessionInfo:
        old = self._sessions[session_id]
        path = await old.get_path_to_root(leaf_id) if leaf_id else await old.load_all()
        info = await self.create(cwd=self._infos[session_id].cwd, name=name, parent_session=session_id)
        new = self._sessions[info.id]
        for entry in path:
            await new.append(entry)
        return info

@hookimpl
def get_store():
    return ('memory', MemoryStore)